In [1]:
import hashlib
import json
import os
import re
from datetime import datetime, timezone
from pathlib import Path

import requests
from cryptography.hazmat.primitives import padding
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes

In [2]:
def json_compact(obj: dict | list) -> str:
    return json.dumps(obj, separators=(',', ':'))

In [3]:
from urllib.parse import urljoin

EAPI_HOST = "https://interfacepc.music.163.com/"
EAPI_KEY = b"e82ckenh8dichen8"
EAPI_SEP = "-36cd479b6b5-"
EAPI_DIGEST = "nobody{}use{}md5forencrypt"

In [4]:
"{}{EAPI_SEP}{}{EAPI_SEP}{}".format("1", "2", "3")

KeyError: 'EAPI_SEP'

In [5]:
f"{}{EAPI_SEP}{}{EAPI_SEP}{}".format("1", "2", "3")

SyntaxError: f-string: valid expression required before '}' (3330657546.py, line 1)

In [6]:
"{}{}{}{}{}".format("{}", EAPI_SEP, "{}", EAPI_SEP, "{}").format("1", "2", "3")

'1-36cd479b6b5-2-36cd479b6b5-3'

In [7]:
EAPI_PLAINTEXT = "{}{}{}{}{}".format("{}", EAPI_SEP, "{}", EAPI_SEP, "{}")

In [8]:
def eapi_encrypt(path: str, payload: dict[str, Any]) -> str:

    padder = padding.PKCS7(128).padder()
    encryptor = Cipher(algorithms.AES(EAPI_KEY), modes.ECB()).encryptor()

    text = json_compact(payload)
    digest = hashlib.md5(EAPI_DIGEST.format(path, text).encode("utf-8")).hexdigest()
    dec = EAPI_PLAINTEXT.format(path, text, digest).encode("utf-8")

    enc = padder.update(dec) + padder.finalize()
    enc = encryptor.update(enc) + encryptor.finalize()
    return enc.hex().upper()

In [ ]:
r = requests.post(
    urljoin(EAPI_HOST, "/eapi/song/enhance/player/url"),
    data={"params": eapi_encrypt("/eapi/song/enhance/player/url", {"ids": [123], "level": "standard", "encodeType": "aac"})},
)

In [9]:
path = "/eapi/usertool/sound/queryAllSound"
r = requests.post(
    urljoin(EAPI_HOST, path),
    data={"params": eapi_encrypt(path, {"exposeType": 5})},
)
r

<Response [200]>

In [10]:
r.content

b'{"code":404,"message":"\xe6\x8e\xa5\xe5\x8f\xa3\xe6\x9c\xaa\xe6\x89\xbe\xe5\x88\xb0\xef\xbc\x81"}'

In [11]:
path = "/eapi/usertool/sound/queryAllSound"
r = requests.post(
    urljoin(EAPI_HOST, path),
    data={
        "params": eapi_encrypt(
            path,
            {
                "exposeType": 5,
                "header": '{"os":"pc","appver":"3.1.29.205117","deviceId":"f64799a1c2461d2d0d4123dee30d1e11","requestId":0,"osver":"Windows 11"}',
            },
        )
    },
)
r.content

b'{"code":404,"message":"\xe6\x8e\xa5\xe5\x8f\xa3\xe6\x9c\xaa\xe6\x89\xbe\xe5\x88\xb0\xef\xbc\x81"}'

In [ ]:
eapi_encrypt(
    path,
    {
        "exposeType": 5,
        "header": '{"os":"pc","appver":"3.1.29.205117","deviceId":"f64799a1c2461d2d0d4123dee30d1e11","requestId":0,"osver":"Windows 11"}',
    },
)

'16380716C2EEC044F8B736481C54EC876B15397605C36214473FBDB495433133710B3E11E380B558129F4DACA9261ABB5E2E34F5695C7AD70252A3A766D7AE666938AD9B38E451B678561419F8D0E324462EE3AF676D09A9087B13263A9C7EBC232C7B2725EE3F045A6C9BD00000E0E3D4A319B8B3D385EA99FF45AD0CDBB63773BB3613EA1C2D0C0EBBF2272BC2529325E8D307838C0079CDCB2C9BF59CA85AF58AD818F8F6773A07A616A131CB233273F5587B87E27E575B7607FBC6C1E33A569AB3101FCBF776AC2D4C5ABBFA06256E3C254E4088CBDE46FAF3140D830D0057895526E4DE404A25B7B80B6230302AEB1E62B20EFB239012B36BFA2098D7B4'

In [13]:
eapi_encrypt(
    path.replace("/eapi/", "/api/"),
    {
        "exposeType": 5,
        "header": '{"os":"pc","appver":"3.1.29.205117","deviceId":"f64799a1c2461d2d0d4123dee30d1e11","requestId":0,"osver":"Windows 11"}',
    },
)

'85195D88B26BCB38D657CA64602A67CCFFE8DB339712557FDF1366EE578829BB17EC032213FDE8D8E640215F72FFCE63A131ACFE429FF1288938961539226D170E6333B469933B4B240C52C530BFA219F3D576BA1BC93FE82B0716F4D09D743C9EEA35FCC73AD8381364A30A058E4F4AEFB2BA80A016CC31F07B394BA59301E752022A5872799582B79516E052593AAF168166A8480277741C0D693EA696C7987349FCCB084BE8C619872708C0FC7D5E9176D814F7DB6CC47886E8BC04115A5FE6E91F9B0F4E814CD2373F9E0C0E474A083586E6A1C5C2E219FA41D5963C9E0AD9B694CC83FD878A7B8EBFB53F3CEE7587C6789BF38D3E3AC0DC10DDFBF82E64'

In [14]:
path = "/eapi/usertool/sound/queryAllSound"
r = requests.post(
    urljoin(EAPI_HOST, path),
    data={
        "params": eapi_encrypt(
            path.replace("/eapi/", "/api/"),
            {
                "exposeType": 5,
                "header": '{"os":"pc","appver":"3.1.29.205117","deviceId":"f64799a1c2461d2d0d4123dee30d1e11","requestId":0,"osver":"Windows 11"}',
            },
        )
    },
)
r.content

b'{"code":200,"data":[{"id":200,"name":"\xe6\x99\xba\xe8\x83\xbd\xe9\x9f\xb3\xe6\x95\x88","subTitle":"\xe6\x99\xba\xe8\x83\xbd\xe5\x8c\xb9\xe9\x85\x8d\xe6\x9c\x80\xe4\xbd\xb3\xe9\x9f\xb3\xe6\x95\x88\xe5\x8f\x82\xe6\x95\xb0\xef\xbc\x8c\xe5\xb8\xa6\xe6\x9d\xa5\xe6\x9e\x81\xe8\x87\xb4\xe5\x90\xac\xe6\xad\x8c\xe4\xbd\x93\xe9\xaa\x8c","playerFloatLayerDesc":"\xe6\x99\xba\xe8\x83\xbd\xe8\xb0\x83\xe5\x8f\x82\xef\xbc\x8c\xe5\x8a\xa8\xe6\x80\x81\xe4\xbc\x98\xe5\x8c\x96\xe9\x9f\xb3\xe8\x89\xb2\xe5\xb9\xb3\xe8\xa1\xa1\xe4\xb8\x8e\xe5\xb1\x82\xe6\xac\xa1","useCount":0,"trackCount":0,"type":3,"newVipStrategyCode":null,"freeBeginTime":null,"freeEndTime":null,"soundType":4,"animation":0,"size":0,"md5":" ","link":" ","version":" ","imgUrl":"","newAdd":0,"vipAuthRespItem":null,"soundMaterialList":null,"linkUrl":null,"soundCategory":4,"listDescText":null,"darkBaseImg":null,"lightBaseImg":null,"darkImg":"https://p5.music.126.net/obj/wo7DmMOIwrXClD_DgMK-/78242608387/5756/80b9/894f/b906580379d50aaac01f6010

In [15]:
path = "/eapi/usertool/sound/queryAllSound"
r = requests.post(
    urljoin(EAPI_HOST, path),
    data={
        "params": eapi_encrypt(
            path.replace("/eapi/", "/api/"),
            {"exposeType": 5},
        )
    },
)
r.content

b'{"code":200,"data":[{"id":200,"name":"\xe6\x99\xba\xe8\x83\xbd\xe9\x9f\xb3\xe6\x95\x88","subTitle":"\xe6\x99\xba\xe8\x83\xbd\xe5\x8c\xb9\xe9\x85\x8d\xe6\x9c\x80\xe4\xbd\xb3\xe9\x9f\xb3\xe6\x95\x88\xe5\x8f\x82\xe6\x95\xb0\xef\xbc\x8c\xe5\xb8\xa6\xe6\x9d\xa5\xe6\x9e\x81\xe8\x87\xb4\xe5\x90\xac\xe6\xad\x8c\xe4\xbd\x93\xe9\xaa\x8c","playerFloatLayerDesc":"\xe6\x99\xba\xe8\x83\xbd\xe8\xb0\x83\xe5\x8f\x82\xef\xbc\x8c\xe5\x8a\xa8\xe6\x80\x81\xe4\xbc\x98\xe5\x8c\x96\xe9\x9f\xb3\xe8\x89\xb2\xe5\xb9\xb3\xe8\xa1\xa1\xe4\xb8\x8e\xe5\xb1\x82\xe6\xac\xa1","useCount":0,"trackCount":0,"type":3,"newVipStrategyCode":null,"freeBeginTime":null,"freeEndTime":null,"soundType":4,"animation":0,"size":0,"md5":" ","link":" ","version":" ","imgUrl":"","newAdd":0,"vipAuthRespItem":null,"soundMaterialList":null,"linkUrl":null,"soundCategory":4,"listDescText":null,"darkBaseImg":null,"lightBaseImg":null,"darkImg":"https://p5.music.126.net/obj/wo7DmMOIwrXClD_DgMK-/78242608387/5756/80b9/894f/b906580379d50aaac01f6010

In [16]:
def eapi_post(path: str, data: dict[str, Any]) -> dict[str, Any]:
    r = requests.post(
        urljoin(EAPI_HOST, "/eapi/" + path.removeprefix("/api/")),
        data={"params": eapi_encrypt(path, data)},
    )
    return r.json()

In [17]:
eapi_post("/api/usertool/sound/queryAllSound", {"exposeType": 5})

{'code': 200,
 'data': [{'id': 200,
   'name': '智能音效',
   'subTitle': '智能匹配最佳音效参数，带来极致听歌体验',
   'playerFloatLayerDesc': '智能调参，动态优化音色平衡与层次',
   'useCount': 0,
   'trackCount': 0,
   'type': 3,
   'newVipStrategyCode': None,
   'freeBeginTime': None,
   'freeEndTime': None,
   'soundType': 4,
   'animation': 0,
   'size': 0,
   'md5': ' ',
   'link': ' ',
   'version': ' ',
   'imgUrl': '',
   'newAdd': 0,
   'vipAuthRespItem': None,
   'soundMaterialList': None,
   'linkUrl': None,
   'soundCategory': 4,
   'listDescText': None,
   'darkBaseImg': None,
   'lightBaseImg': None,
   'darkImg': 'https://p5.music.126.net/obj/wo7DmMOIwrXClD_DgMK-/78242608387/5756/80b9/894f/b906580379d50aaac01f60102b8d87b1.png',
   'lightImg': 'https://p5.music.126.net/obj/wo7DmMOIwrXClD_DgMK-/78242612936/6936/c317/53a1/6f70b0aae1c43ef74f551866c8b2d4da.png',
   'darkGif': 'https://p5.music.126.net/obj/wo7DmMOIwrXClD_DgMK-/78242608387/5756/80b9/894f/b906580379d50aaac01f60102b8d87b1.png',
   'lightGif': 'https:/

In [18]:
eapi_post("/api/usertool/sound/queryAllSound", {"exposeType": 6})

{'code': 200,
 'data': [{'id': 3592400,
   'name': '8D环绕',
   'subTitle': '声音动态旋转，享受音乐从颅内穿梭而过的律动',
   'playerFloatLayerDesc': '声音动态旋转，享受穿颅而过的律动',
   'useCount': 0,
   'trackCount': 0,
   'type': 4,
   'newVipStrategyCode': None,
   'freeBeginTime': None,
   'freeEndTime': None,
   'soundType': 3,
   'animation': 0,
   'size': 0,
   'md5': 'b772477b6d9e0fa15aa94eb9d6e2b0ed',
   'link': 'http://d1.music.126.net/dmusic/obj/wo_Dk8OFw6LCn2vCm8Kx/62737629769/6908/0757/5235/b772477b6d9e0fa15aa94eb9d6e2b0ed.ncae',
   'version': ' ',
   'imgUrl': '',
   'newAdd': 0,
   'vipAuthRespItem': None,
   'soundMaterialList': [{'size': 253,
     'md5': 'b772477b6d9e0fa15aa94eb9d6e2b0ed',
     'link': 'http://d1.music.126.net/dmusic/obj/wo_Dk8OFw6LCn2vCm8Kx/62737629769/6908/0757/5235/b772477b6d9e0fa15aa94eb9d6e2b0ed.ncae',
     'type': 1}],
   'linkUrl': None,
   'soundCategory': 3,
   'listDescText': None,
   'darkBaseImg': None,
   'lightBaseImg': None,
   'darkImg': 'https://p5.music.126.net/obj/wo7Dm

In [19]:
eapi_post("/api/usertool/sound/queryAllSound", {"exposeType": 7})

{'code': 200,
 'data': [{'id': 3592428,
   'name': '3D环绕',
   'subTitle': '戴上耳机，沉浸式拥抱四方涌来的音乐',
   'playerFloatLayerDesc': '声音从四方传来，耳朵被音乐包裹',
   'useCount': 0,
   'trackCount': 0,
   'type': 1,
   'newVipStrategyCode': None,
   'freeBeginTime': None,
   'freeEndTime': None,
   'soundType': 3,
   'animation': 0,
   'size': 0,
   'md5': '403af171cb2f8af205925e20741a3913',
   'link': 'http://d1.music.126.net/dmusic/obj/wo_Dk8OFw6LCn2vCm8Kx/78242522013/cd70/7ee1/908c/403af171cb2f8af205925e20741a3913.ncae',
   'version': ' ',
   'imgUrl': '',
   'newAdd': 0,
   'vipAuthRespItem': None,
   'soundMaterialList': [{'size': 373,
     'md5': '403af171cb2f8af205925e20741a3913',
     'link': 'http://d1.music.126.net/dmusic/obj/wo_Dk8OFw6LCn2vCm8Kx/78242522013/cd70/7ee1/908c/403af171cb2f8af205925e20741a3913.ncae',
     'type': 1}],
   'linkUrl': None,
   'soundCategory': 3,
   'listDescText': None,
   'darkBaseImg': None,
   'lightBaseImg': None,
   'darkImg': None,
   'lightImg': None,
   'darkGif':

In [21]:
eapi_post("/api/usertool/sound/queryAllSound", {"exposeType": 1})

{'code': 200,
 'data': [{'id': 100,
   'name': '智能音效',
   'subTitle': '智能匹配最佳音效参数，带来极致听歌体验',
   'playerFloatLayerDesc': '智能调参，动态优化音色平衡与层次',
   'useCount': 0,
   'trackCount': 0,
   'type': 3,
   'newVipStrategyCode': None,
   'freeBeginTime': None,
   'freeEndTime': None,
   'soundType': 2,
   'animation': 0,
   'size': 0,
   'md5': None,
   'link': ' ',
   'version': ' ',
   'imgUrl': '',
   'newAdd': 0,
   'vipAuthRespItem': None,
   'soundMaterialList': None,
   'linkUrl': None,
   'soundCategory': 2,
   'listDescText': None,
   'darkBaseImg': None,
   'lightBaseImg': None,
   'darkImg': 'https://p5.music.126.net/obj/wovDmcKXw6PCn2_CmsOk/62306614368/af34/6fb5/616d/d978c8cb8854ea0085954d7cef6ab07a.png',
   'lightImg': 'https://p5.music.126.net/obj/wo7DmMOIwrXClD_DgMK-/62178097417/8e71/78d7/565e/809e17609bfe15d097f39c63aed1e89a.png',
   'darkGif': 'https://p5.music.126.net/obj/wovDmcKXw6PCn2_CmsOk/62669917653/c53e/ef88/b814/65b7895c2b312f40c112589a784bb6d9.webp',
   'lightGif': 'https

In [22]:
eapi_post("/api/usertool/sound/queryAllSound", {"exposeType": 2})

{'code': 200,
 'data': [{'id': 3601400,
   'name': '华晨宇音效',
   'subTitle': '专属定制，适合听花花的歌曲',
   'playerFloatLayerDesc': '专属定制，适合听花花的歌曲',
   'useCount': 1509,
   'trackCount': 0,
   'type': 3,
   'newVipStrategyCode': None,
   'freeBeginTime': None,
   'freeEndTime': None,
   'soundType': 1,
   'animation': 0,
   'size': 0,
   'md5': '7e37a01730eb71e01186f9c906621c27',
   'link': 'http://d1.music.126.net/dmusic/obj/w57Cg8KTwrbClD_CmMK-/82462197821/3d71/0e57/b8fb/4ffa0d4c4d303569606a6905b6b4ec3d.ncae',
   'version': ' ',
   'imgUrl': '',
   'newAdd': 0,
   'vipAuthRespItem': None,
   'soundMaterialList': [{'size': 649,
     'md5': '4ffa0d4c4d303569606a6905b6b4ec3d',
     'link': 'http://d1.music.126.net/dmusic/obj/w57Cg8KTwrbClD_CmMK-/82462197821/3d71/0e57/b8fb/4ffa0d4c4d303569606a6905b6b4ec3d.ncae',
     'type': 1}],
   'linkUrl': None,
   'soundCategory': 1,
   'listDescText': None,
   'darkBaseImg': None,
   'lightBaseImg': None,
   'darkImg': 'https://p5.music.126.net/obj/wovDmcKXw6PC

In [23]:
eapi_post("/api/usertool/sound/queryAllSound", {"exposeType": 3})

{'code': 200,
 'data': [{'id': 3600400,
   'name': '全景环绕',
   'subTitle': '戴上耳机，人声在耳边，乐器在身后，环境声从四周涌来，像坐在舞台中间听一场私人 Live。',
   'playerFloatLayerDesc': '闭上眼睛，声音在你身边转动',
   'useCount': 428052,
   'trackCount': 0,
   'type': 3,
   'newVipStrategyCode': None,
   'freeBeginTime': None,
   'freeEndTime': None,
   'soundType': 1,
   'animation': 0,
   'size': 0,
   'md5': 'd01b1da78d594d5dee0e0d19f070fa3d',
   'link': 'http://d1.music.126.net/dmusic/obj/w57Cg8KTwrbClD_CmMK-/80952664736/b695/7fc2/8b31/9564dd306959d052bda5694f44fb75b5.ncae',
   'version': ' ',
   'imgUrl': '',
   'newAdd': 0,
   'vipAuthRespItem': None,
   'soundMaterialList': [{'size': 375,
     'md5': '9564dd306959d052bda5694f44fb75b5',
     'link': 'http://d1.music.126.net/dmusic/obj/w57Cg8KTwrbClD_CmMK-/80952664736/b695/7fc2/8b31/9564dd306959d052bda5694f44fb75b5.ncae',
     'type': 1}],
   'linkUrl': None,
   'soundCategory': 1,
   'listDescText': None,
   'darkBaseImg': None,
   'lightBaseImg': None,
   'darkImg': None,
   '

In [24]:
eapi_post("/api/usertool/sound/queryAllSound", {"exposeType": 4})

{'code': 200, 'data': [], 'message': ''}

In [25]:
eapi_post("/api/usertool/sound/queryAllSound", {"exposeType": 8})

{'code': 200, 'data': [], 'message': ''}